# Modul 06: Datenaufteilung, Baselines, Verluste und Metriken | Lösungen

## Überblick

Sie erstellen reproduzierbare Train-, Validierungs- und Testaufteilungen und berücksichtigen dabei Klassenverteilung, Gruppen und Zeitordnung. Anschließend berechnen Sie Regressions- und Klassifikationsmetriken manuell, vergleichen Baselines und erkennen typische Datenleckage.

**Zugehörige Vorlesungen**

- **Daten aufteilen**
- **Verluste und Metriken**

## Lernziele

Nach der Bearbeitung können Sie:

- reproduzierbare Splits mit Indizes erstellen und Sonderstrukturen wie Klassen, Gruppen und Zeit berücksichtigen.
- Datenleckage durch Zielinformationen, überlappende Gruppen oder falsch angepasste Vorverarbeitung erkennen.
- Regressionsverluste, Baselines, Konfusionsmatrix, Precision, Recall, F1 und probabilistische Verluste berechnen.

## Geprüfte Fähigkeiten

- Indexbasierte, stratifizierte, gruppenbasierte und zeitliche Splits
- leakage-freie Skalierung und dokumentierte Splitkontrollen
- manuelle Metrikberechnung und fachlich passende Baselines

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** mittel
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle erzeugt kleine tabellarische Daten mit wiederholten Patientengruppen, Zeitordnung und unausgewogener Zielvariable. Zusätzlich stehen feste Regressions- und Klassifikationsvorhersagen bereit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# 45 Gruppen mit je drei wiederholten Messungen.
anzahl_gruppen = 45
messungen_pro_gruppe = 3
gruppen_id = np.repeat(np.arange(1001, 1001 + anzahl_gruppen), messungen_pro_gruppe)
zeitpunkt = np.tile(np.arange(messungen_pro_gruppe), anzahl_gruppen)

alter_basis = rng.integers(25, 75, size=anzahl_gruppen)
alter = np.repeat(alter_basis, messungen_pro_gruppe) + zeitpunkt
marker = rng.normal(0, 1, size=len(gruppen_id)) + 0.025 * (alter - 50)
puls = rng.normal(72, 8, size=len(gruppen_id)) + 4 * marker

# Seltenes positives Ereignis, das von Alter und Marker abhängt.
logit = -2.5 + 0.045 * (alter - 50) + 0.9 * marker
wahrscheinlichkeit = 1 / (1 + np.exp(-logit))
ziel = rng.binomial(1, wahrscheinlichkeit)

patienten = pd.DataFrame(
    {
        "gruppen_id": gruppen_id,
        "zeitpunkt": zeitpunkt,
        "alter": alter.astype(float),
        "marker": marker,
        "puls": puls,
        "ziel": ziel,
    }
)

# Feste Regressionsdaten für manuelle Verlustberechnungen.
y_reg_true = np.array([10.0, 12.0, 14.0, 18.0, 25.0, 40.0])
y_reg_model = np.array([11.0, 11.5, 16.0, 17.0, 23.0, 31.0])

# Feste Wahrscheinlichkeiten für eine binäre Klassifikation.
y_cls_true = np.array([0, 0, 1, 1, 0, 1, 0, 1, 1, 0])
y_cls_score = np.array([0.10, 0.35, 0.62, 0.91, 0.55, 0.74, 0.20, 0.48, 0.83, 0.05])

print("Einrichtung abgeschlossen.")
print("Patiententabelle:", patienten.shape)
print("Positive Klasse:", f"{patienten['ziel'].mean():.1%}")

### Aufgabe 1: Train-, Validierungs- und Testindizes manuell erzeugen

Verwenden Sie die Zeilenindizes von `patienten`:

1. Mischen Sie die Indizes reproduzierbar mit einem lokalen Zufallsgenerator.
2. Teilen Sie 60 % Training, 20 % Validierung und 20 % Test zu.
3. Prüfen Sie Splitgrößen, vollständige Abdeckung und fehlende Überschneidungen.
4. Speichern Sie die Teilmengen als `train_manuell`, `valid_manuell` und `test_manuell`.
5. Dokumentieren Sie den verwendeten Seed und die tatsächlichen Größen in einem DataFrame.

In [ ]:
alle_indizes = patienten.index.to_numpy()

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Ein lokaler Generator verhindert, dass andere Zufallsaufrufe den Split verändern.
split_rng = np.random.default_rng(RANDOM_SEED)
gemischte_indizes = split_rng.permutation(alle_indizes)

anzahl = len(gemischte_indizes)
train_ende = int(0.60 * anzahl)
valid_ende = int(0.80 * anzahl)

train_idx = gemischte_indizes[:train_ende]
valid_idx = gemischte_indizes[train_ende:valid_ende]
test_idx = gemischte_indizes[valid_ende:]

# Mengenprüfungen machen Überschneidungen und verlorene Zeilen sichtbar.
assert set(train_idx).isdisjoint(valid_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(valid_idx).isdisjoint(test_idx)
assert set(np.concatenate([train_idx, valid_idx, test_idx])) == set(alle_indizes)

train_manuell = patienten.loc[train_idx].copy()
valid_manuell = patienten.loc[valid_idx].copy()
test_manuell = patienten.loc[test_idx].copy()

split_protokoll = pd.DataFrame(
    {
        "Split": ["Training", "Validierung", "Test"],
        "Zeilen": [len(train_manuell), len(valid_manuell), len(test_manuell)],
        "Anteil": [len(train_manuell) / anzahl, len(valid_manuell) / anzahl, len(test_manuell) / anzahl],
        "Seed": [RANDOM_SEED] * 3,
    }
)
display(split_protokoll)

> **Musterantwort und Interpretation**
>
> Er berücksichtigt nicht, dass mehrere Zeilen zur selben Patientengruppe gehören. Dadurch können Messungen derselben Person in Training und Test landen. Das würde eine unrealistisch leichte Bewertung erzeugen, wenn personenspezifische Muster wiedererkannt werden.

### Aufgabe 2: Stratifizierte, gruppenbasierte und zeitliche Splits vergleichen

Erstellen Sie drei alternative Splits:

1. **Stratifiziert:** 75 % Training und 25 % Test mit ähnlichem Klassenanteil.
2. **Gruppenbasiert:** ganze `gruppen_id` im Verhältnis ungefähr 75/25 trennen.
3. **Zeitlich:** alle Zeilen mit `zeitpunkt < 2` als Training und `zeitpunkt == 2` als Test.

Berichten Sie für jeden Split Zeilenzahl, positiven Anteil und gegebenenfalls Anzahl überlappender Gruppen. Erklären Sie, für welche Einsatzannahme jeder Split geeignet ist.

In [ ]:
merkmale = ["alter", "marker", "puls"]
X_patienten = patienten[merkmale]
y_patienten = patienten["ziel"]

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# 1. Stratifizierung erhält den Zielklassenanteil, ignoriert aber Gruppenabhängigkeiten.
strat_train_idx, strat_test_idx = train_test_split(
    patienten.index,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_patienten,
)

# 2. GroupShuffleSplit hält alle Zeilen einer Gruppe vollständig zusammen.
gruppen_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
grupp_train_pos, grupp_test_pos = next(
    gruppen_splitter.split(X_patienten, y_patienten, groups=patienten["gruppen_id"])
)
grupp_train_idx = patienten.index.to_numpy()[grupp_train_pos]
grupp_test_idx = patienten.index.to_numpy()[grupp_test_pos]

# 3. Zeitlicher Split bewahrt die Reihenfolge und simuliert die Vorhersage eines späteren Zeitpunkts.
zeit_train_idx = patienten.index[patienten["zeitpunkt"] < 2].to_numpy()
zeit_test_idx = patienten.index[patienten["zeitpunkt"] == 2].to_numpy()


def split_zeile(name, train_index, test_index):
    train_daten = patienten.loc[train_index]
    test_daten = patienten.loc[test_index]
    gruppen_ueberlappung = len(
        set(train_daten["gruppen_id"]) & set(test_daten["gruppen_id"])
    )
    return {
        "Split": name,
        "Train_Zeilen": len(train_daten),
        "Test_Zeilen": len(test_daten),
        "Train_Positive": train_daten["ziel"].mean(),
        "Test_Positive": test_daten["ziel"].mean(),
        "Überlappende_Gruppen": gruppen_ueberlappung,
    }


split_vergleich = pd.DataFrame(
    [
        split_zeile("stratifiziert", strat_train_idx, strat_test_idx),
        split_zeile("gruppenbasiert", grupp_train_idx, grupp_test_idx),
        split_zeile("zeitlich", zeit_train_idx, zeit_test_idx),
    ]
)
display(split_vergleich.round(3))

# Der gruppenbasierte Split darf keine gemeinsame Gruppen-ID besitzen.
assert split_vergleich.loc[split_vergleich["Split"] == "gruppenbasiert", "Überlappende_Gruppen"].iloc[0] == 0

> **Musterantwort und Interpretation**
>
> Ein stratifizierter Split passt zu unabhängigen Beobachtungen, wenn nur die Klassenverteilung stabil gehalten werden soll. Ein gruppenbasierter Split prüft die Übertragung auf vollständig neue Personen oder Anlagen. Ein zeitlicher Split simuliert einen späteren Einsatzzeitraum und schützt vor Zukunftsinformationen.

### Aufgabe 3: Datenleckage durch Zielinformation und Skalierung erkennen

1. Erzeugen Sie absichtlich ein unzulässiges Merkmal `ziel_kopie = ziel` und erklären Sie, warum es Zielwertleckage ist.
2. Verwenden Sie den gruppenbasierten Split aus Aufgabe 2.
3. Vergleichen Sie zwei Skalierungen:
   - falsch: `StandardScaler` auf allen Daten anpassen,
   - korrekt: nur auf Trainingsdaten anpassen und danach Testdaten transformieren.
4. Geben Sie die gelernten Mittelwerte beider Skalierer aus.
5. Trainieren Sie ein logistisches Modell ausschließlich mit den korrekten Merkmalen und der korrekten Skalierung.

In [ ]:
patienten_leakage = patienten.copy()
patienten_leakage["ziel_kopie"] = patienten_leakage["ziel"]

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Eine Kopie des Zielwerts verrät die richtige Antwort direkt und darf nicht in X enthalten sein.
assert np.array_equal(patienten_leakage["ziel_kopie"], patienten_leakage["ziel"])

X_train = patienten.loc[grupp_train_idx, merkmale]
X_test = patienten.loc[grupp_test_idx, merkmale]
y_train = patienten.loc[grupp_train_idx, "ziel"]
y_test = patienten.loc[grupp_test_idx, "ziel"]

# FALSCH: Der Skalierer sieht bereits die Verteilung der Testdaten.
skalierer_falsch = StandardScaler()
skalierer_falsch.fit(patienten[merkmale])

# KORREKT: Alle gelernten Parameter stammen ausschließlich aus Trainingsdaten.
skalierer_korrekt = StandardScaler()
X_train_skaliert = skalierer_korrekt.fit_transform(X_train)
X_test_skaliert = skalierer_korrekt.transform(X_test)

print("Mittelwerte, falsch auf allen Daten gelernt:", np.round(skalierer_falsch.mean_, 3))
print("Mittelwerte, korrekt nur im Training gelernt:", np.round(skalierer_korrekt.mean_, 3))

modell = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
modell.fit(X_train_skaliert, y_train)
vorhersage = modell.predict(X_test_skaliert)
print(f"Testgenauigkeit ohne Zielwertleckage: {accuracy_score(y_test, vorhersage):.3f}")

> **Musterantwort und Interpretation**
>
> Auch Mittelwerte, Standardabweichungen, Imputationswerte oder Kategorienhäufigkeiten sind aus Daten gelernte Informationen. Werden sie unter Einbeziehung des Tests bestimmt, beeinflusst die spätere Testverteilung bereits den Trainingsprozess und die Bewertung wird zu optimistisch.

### Aufgabe 4: Regressionsverluste und konstante Baselines manuell berechnen

Berechnen Sie ausschließlich mit NumPy:

1. Residuen `Vorhersage - Istwert`.
2. MAE, MSE und RMSE für `y_reg_model`.
3. Eine Mittelwert-Baseline, deren Konstante nur aus den ersten vier Zielwerten gelernt wird.
4. Eine Median-Baseline mit derselben Trainingsmenge.
5. MAE und RMSE beider Baselines auf den letzten zwei Werten.
6. Eine Vergleichstabelle und eine kurze Interpretation der großen letzten Abweichung.

In [ ]:
y_reg_train = y_reg_true[:4]
y_reg_test = y_reg_true[4:]
modell_test_vorhersage = y_reg_model[4:]

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Residuen behalten die Fehlerrichtung; positive Werte bedeuten Überschätzung.
residuen_gesamt = y_reg_model - y_reg_true
mae_gesamt = np.mean(np.abs(residuen_gesamt))
mse_gesamt = np.mean(residuen_gesamt ** 2)
rmse_gesamt = np.sqrt(mse_gesamt)

print("Residuen:", residuen_gesamt)
print(f"Modell MAE gesamt: {mae_gesamt:.3f}")
print(f"Modell MSE gesamt: {mse_gesamt:.3f}")
print(f"Modell RMSE gesamt: {rmse_gesamt:.3f}")

# Baseline-Konstanten dürfen nur aus den Trainingszielen stammen.
mittelwert_konstante = y_reg_train.mean()
median_konstante = np.median(y_reg_train)
mittelwert_pred = np.full_like(y_reg_test, mittelwert_konstante, dtype=float)
median_pred = np.full_like(y_reg_test, median_konstante, dtype=float)


def reg_kennzahlen(y_true, y_pred):
    fehler = y_pred - y_true
    return np.mean(np.abs(fehler)), np.sqrt(np.mean(fehler ** 2))


vergleich = []
for name, pred in [
    ("Beispielmodell", modell_test_vorhersage),
    ("Mittelwert-Baseline", mittelwert_pred),
    ("Median-Baseline", median_pred),
]:
    mae, rmse = reg_kennzahlen(y_reg_test, pred)
    vergleich.append({"Modell": name, "MAE": mae, "RMSE": rmse, "Vorhersagen": pred.tolist()})

vergleich_reg = pd.DataFrame(vergleich)
display(vergleich_reg.round(3))

> **Musterantwort und Interpretation**
>
> Beim RMSE werden Fehler vor dem Mitteln quadriert. Ein Fehler von 9 trägt daher 81 zum MSE bei und erhält deutlich mehr Gewicht als mehrere kleine Fehler. Der MAE verwendet dagegen absolute Abstände und bleibt näher an einer durchschnittlichen Fehlergröße in Zieleinheiten.

### Aufgabe 5: Konfusionsmatrix, Schwellenwerte und Wahrscheinlichkeitsverlust

Schreiben Sie eine Funktion `klassifikationsbericht(y_true, scores, schwelle)`, die manuell berechnet:

- vorhergesagte Labels,
- TP, TN, FP und FN,
- Accuracy, Precision, Recall und F1,
- binäre Kreuzentropie mit numerischer Absicherung.

Vergleichen Sie die Schwellenwerte 0,50 und 0,70 in einer Tabelle.

In [ ]:
def klassifikationsbericht(y_true, scores, schwelle):
    """Berechnet binäre Kennzahlen ohne sklearn-Metrikfunktionen."""
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def klassifikationsbericht(y_true, scores, schwelle):
    """Berechnet binäre Kennzahlen ohne sklearn-Metrikfunktionen."""
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    y_pred = (scores >= schwelle).astype(int)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    # clipping verhindert log(0), ohne die Rangfolge der Wahrscheinlichkeiten praktisch zu verändern.
    eps = 1e-12
    p = np.clip(scores, eps, 1 - eps)
    log_loss = -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

    return {
        "Schwelle": schwelle,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Kreuzentropie": log_loss,
    }


berichte = pd.DataFrame(
    [
        klassifikationsbericht(y_cls_true, y_cls_score, 0.50),
        klassifikationsbericht(y_cls_true, y_cls_score, 0.70),
    ]
)
display(berichte.round(3))

> **Musterantwort und Interpretation**
>
> Die Kreuzentropie bewertet die ursprünglichen Wahrscheinlichkeiten direkt und verwendet keine harte Klassenentscheidung. Der Schwellenwert verändert TP, FP, FN und TN, aber nicht die bereits ausgegebenen Scores. Für die Wahl eines Schwellenwerts müssen daher Fehlerfolgen und schwellenabhängige Metriken betrachtet werden.

### Aufgabe 6: Integrationsaufgabe: gruppensichere Bewertung mit Baseline

Nutzen Sie den gruppenbasierten Split und erstellen Sie einen vollständigen kleinen Bewertungsablauf:

1. Mehrheitsklassen-Baseline aus `y_train`.
2. Leakage-freie Skalierung.
3. Logistische Regression.
4. Wahrscheinlichkeiten und Vorhersagen bei Schwelle 0,50.
5. Manuelle Kennzahlen mit Ihrer Funktion.
6. Vergleich mit der Baseline.
7. Splitprotokoll mit Gruppenanzahl, Klassenanteilen und Seed.

Begründen Sie, ob Accuracy bei der vorliegenden Klassenverteilung ausreicht.

In [ ]:
# X_train, X_test, y_train und y_test stammen aus Aufgabe 3.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Die Baseline lernt nur die häufigste Klasse des Trainingssplits.
mehrheitsklasse = int(y_train.value_counts().idxmax())
baseline_pred = np.full(len(y_test), mehrheitsklasse, dtype=int)
baseline_score = np.full(len(y_test), y_train.mean(), dtype=float)

# Vorverarbeitung wird ausschließlich auf dem Training gelernt.
skalierer = StandardScaler()
X_train_scaled = skalierer.fit_transform(X_train)
X_test_scaled = skalierer.transform(X_test)

modell = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
modell.fit(X_train_scaled, y_train)
modell_scores = modell.predict_proba(X_test_scaled)[:, 1]

baseline_bericht = klassifikationsbericht(y_test.to_numpy(), baseline_score, 0.50)
modell_bericht = klassifikationsbericht(y_test.to_numpy(), modell_scores, 0.50)

vergleich = pd.DataFrame(
    [
        {"Modell": "Mehrheitsbaseline", **baseline_bericht},
        {"Modell": "Logistische Regression", **modell_bericht},
    ]
)
display(vergleich.round(3))

split_dokumentation = pd.DataFrame(
    {
        "Eigenschaft": [
            "Seed",
            "Trainingszeilen",
            "Testzeilen",
            "Trainingsgruppen",
            "Testgruppen",
            "Überlappende Gruppen",
            "Positiver Anteil Training",
            "Positiver Anteil Test",
        ],
        "Wert": [
            RANDOM_SEED,
            len(X_train),
            len(X_test),
            patienten.loc[grupp_train_idx, "gruppen_id"].nunique(),
            patienten.loc[grupp_test_idx, "gruppen_id"].nunique(),
            len(
                set(patienten.loc[grupp_train_idx, "gruppen_id"])
                & set(patienten.loc[grupp_test_idx, "gruppen_id"])
            ),
            y_train.mean(),
            y_test.mean(),
        ],
    }
)
display(split_dokumentation)

> **Musterantwort und Interpretation**
>
> Accuracy allein kann irreführend sein, weil eine Mehrheitsbaseline bereits viele negative Fälle korrekt klassifiziert. Entscheidend ist, ob das Modell positive Fälle besser findet, ohne unvertretbar viele Fehlalarme zu erzeugen. Recall, Precision, F1 und die konkrete Kostenstruktur sollten gemeinsam betrachtet werden. Bei wenigen Testgruppen sind die Werte zudem unsicher und sollten nicht überinterpretiert werden.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?